In [1]:
import sys
import os
import pandas as pd
import numpy as np
import xgboost as xgb

from sklearn.metrics import ndcg_score

# 1つ上のディレクトリにパスを通す
sys.path.append(os.path.abspath('../'))

from module.Dataset import read_preprocess_tsv, read_test_tsv, train_division, ColumnEncode


In [2]:
# 前処理済みデータの読み込み
train_A = read_preprocess_tsv('A')


In [3]:
# 予測用データの読み込み
test_A = read_test_tsv('A')


In [4]:
# エンコードクラスのインスタンス化
user_item_enc = ColumnEncode()

In [5]:
# エンコードの実行
train_A['user_id'] = user_item_enc.user_encode(train_A['user_id'])
train_A['product_id'] = user_item_enc.product_encode(train_A['product_id'])

test_A['user_id'] = user_item_enc.test_encode(test_A['user_id'])

In [6]:
# タイムスタンプを数値型に変換
train_A["time_stamp"] = pd.to_datetime(train_A["time_stamp"])
train_A['unix_time'] = train_A['time_stamp'].astype(np.int64) // 10**9  # UNIX時間に変換


In [7]:
# データ分割
train_X, train_y, valid_X, valid_y, features = train_division(train_A)


In [8]:
# 時系列交差っぽい検証
model = xgb.XGBRegressor(objective='reg:squarederror')

model.fit(train_X, train_y)
predictions = model.predict(valid_X)
print("Validation NDCG:", ndcg_score([valid_y], [predictions]))


Validation NDCG: 0.9999999999999815


In [9]:
model.fit(valid_X, valid_y)
predictions = model.predict(train_X)
print("Validation NDCG:", ndcg_score([train_y], [predictions]))


Validation NDCG: 0.9999999999999857


In [10]:
# 最終学習用データ作成
X_train = train_A[features]
y_train = train_A['relevance']


In [11]:
# 最終学習
model.fit(X_train, y_train)


XGBRegressor(base_score=None, booster=None, callbacks=None,
             colsample_bylevel=None, colsample_bynode=None,
             colsample_bytree=None, device=None, early_stopping_rounds=None,
             enable_categorical=False, eval_metric=None, feature_types=None,
             gamma=None, grow_policy=None, importance_type=None,
             interaction_constraints=None, learning_rate=None, max_bin=None,
             max_cat_threshold=None, max_cat_to_onehot=None,
             max_delta_step=None, max_depth=None, max_leaves=None,
             min_child_weight=None, missing=nan, monotone_constraints=None,
             multi_strategy=None, n_estimators=None, n_jobs=None,
             num_parallel_tree=None, random_state=None, ...)

In [12]:
# 推薦関数
def recommend_products(user_id, top_k=22):
    user_data = train_A[train_A['user_id'] == user_id][features]
    if user_data.empty:
        print(f"Warning: User {user_id} has no data.")
        return pd.DataFrame(columns=['product_id', 'rank'])
    predictions = model.predict(user_data)
    user_data['score'] = predictions
    recommendations = user_data.sort_values(by='score', ascending=False).drop_duplicates(subset=['user_id', 'product_id']).head(top_k)
    recommendations['rank'] = range(1, len(recommendations) + 1)
    recommendations['user_id'] = user_item_enc.user_decode(recommendations['user_id'])
    recommendations['product_id'] = user_item_enc.product_decode(recommendations['product_id'])
    return recommendations[['user_id', 'product_id', 'rank']]


In [13]:
# 推薦結果と実際の結果を取得
def evaluate_ndcg():
    y_true = []
    y_score = []

    for user_id in test_A['user_id'].unique():
        # 実際の関連度 (train_Aのuser_idに基づいて)
        actual = train_A[train_A['user_id'] == user_id].sort_values(by='relevance', ascending=False)['relevance'].values

        # 推薦結果（モデルの予測スコアを基に）
        pred_df = recommend_products(user_id, top_k=len(actual))  # 商品IDとスコアの DataFrame
        if pred_df.empty:
            continue
        pred_ranks = pred_df['rank'].values
        pred_scores = 1 / (pred_ranks + 1e-5)  # 小さいrankほど高スコア
        # 関連度と予測スコアをリストに追加
        y_true.append(actual)
        y_score.append(pred_scores)

    # NDCGを計算
    return np.mean([ndcg_score([t], [s]) for t, s in zip(y_true, y_score)]) if y_true else 0.0


In [14]:
# 予測結果保存
def save_predictions():
    results = []
    for user_id in test_A['user_id'].unique():
        if user_id in train_A['user_id'].values:
            recommendations = recommend_products(user_id)
            for _, row in recommendations.iterrows():
                results.append([row['user_id'], row['product_id'], row['rank']])
    if not results:
        print("Warning: No predictions generated.")
    pd.DataFrame(results, columns=['user_id', 'product_id', 'rank']).to_csv('../dataset/predict_A.tsv', sep='\t', index=False)


In [15]:
# 推薦結果表示
#print("nDCG Score:", evaluate_ndcg())


In [16]:
#save_predictions()
